In [0]:
%run ./downstream_comm

In [0]:
%run ../00_common/data_utils

In [0]:
def get_execute_param(key, default_value):
    try:
        value = dbutils.widgets.get(key)
        if value == None or len(value.strip()) ==0:
            return default_value
        else:
            return value
    except:
        return default_value

In [0]:
def get_config_ids_from_table_name(table_name, condition_str= None):

    config_ids =( [row[0] for row in
            spark.table(f"{config_database}.downstream_config")
            .filter(col("is_active") == True)
            .filter(col("table_name") == table_name)
            .where(condition_str if condition_str != None else "1=1")
            .select("config_id")
            .collect()
        ]
    )

    if len(config_ids) == 0:
        raise ValueError(f"没有对应配置信息. table_name: {table_name}, condition_str: {condition_str}")

    return config_ids




def send_data_by_config_id(task_id, config_id, task_inner_parallelism, first_cdc_operation_time, second_cdc_operation_time, condition_str: str = None):
    
    logging.info("="*120)
    logging.info(f"config_id: {config_id}")

    config_list = get_downstreamConfig_list_from_DF(spark.table(f"{config_database}.downstream_config").filter(col("is_active") == True).filter(col("config_id") == config_id))

    if len(config_list) == 0:
        raise ValueError(f"没有对应配置信息. config_id: {config_id}")

    elif len(config_list) > 1:
        raise ValueError(f"config_id 必须唯一: {config_id}")  

    elif len(config_list) == 1:
        config_obj = config_list[0]
        
        config_obj.task_id = task_id
        config_obj.first_cdc_operation_time = first_cdc_operation_time if first_cdc_operation_time == None else datetime.strptime(first_cdc_operation_time, CDC_OPERATION_TIME_FORMAT)
        config_obj.second_cdc_operation_time = second_cdc_operation_time if second_cdc_operation_time == None else datetime.strptime(second_cdc_operation_time, CDC_OPERATION_TIME_FORMAT)

        if condition_str != None:
            config_obj.condition_str = condition_str

        logging.info(config_obj)

        is_send = send_data_to_downstream(config_obj, task_inner_parallelism)

    return True


def check_first_cdc_operation_time(config_id_list, first_cdc_operation_time, second_cdc_operation_time):
    if first_cdc_operation_time != None:
        if config_id_list == None or  len(config_id_list.split(",")) > 1  :
            raise ValueError("指定 first_cdc_operation_time 参数时, 必须指定 config_id_list 且 config_id_list中只能有一个 config_id !!!")
    else:
        if second_cdc_operation_time != None:
            raise ValueError("指定 second_cdc_operation_time 时必须指定 first_cdc_operation_time!!!")
    

def downstream_job_run(task_id, config_table_name, config_table_condition_str, config_id_list, max_task_num, task_inner_parallelism, first_cdc_operation_time, second_cdc_operation_time, condition_str: str = None):
    '''
        config_table_name:  需要下发数据的表名. 会获取该表名下所有config
        config_table_condition_str: 配合config_table_name 使用, 对该表名下所有config 进行过滤.  格式为sql语句, 例: market = 'AUS'

        config_id_list: 格式为逗号分隔的字符串. 例: a_id,b_id
        如果 config_id_list不为空, 则不会使用 config_table_name 获取config.

        first_cdc_operation_time: 指定增量数据起始版本的 cdc_operation_time, 例: 2024-11-20T08:39:34.476+00:00 
        second_cdc_operation_time: 指定增量数据终止版本的 cdc_operation_time, 例: 2024-11-20T08:39:34.476+00:00
            增量数据取数范围:  first_cdc_operation_time < cdc_operation_time <= second_cdc_operation_time;
            当指定 first_cdc_operation_time, 未指定 second_cdc_operation_time 时, second_cdc_operation_time 为最大 cdc_operation_time;
            指定 second_cdc_operation_time 时必须指定 first_cdc_operation_time;
            指定 first_cdc_operation_time 参数时, 必须指定 config_id_list 且 config_id_list中只能有一个 config_id.
        
        condition_str: 覆盖配置表中数据过滤条件
    '''

    check_first_cdc_operation_time(config_id_list, first_cdc_operation_time, second_cdc_operation_time)

    config_ids = []

    if config_id_list != None:
        config_ids = [config_id.strip() for config_id in  config_id_list.split(",")]

    elif config_table_name != None:
        config_ids = get_config_ids_from_table_name(config_table_name, config_table_condition_str)
    
    else:
        raise ValueError("config_id_list 和 config_table_name 必须指定一个.")
    
    
    logging.info(config_ids)

    if len(config_ids) < max_task_num:
        max_task_num = len(config_ids)

    except_list = []

    # 开启线程池
    with ThreadPoolExecutor(max_workers=max_task_num) as executor:
        future_to_success = {executor.submit(send_data_by_config_id, task_id, config_id, task_inner_parallelism, first_cdc_operation_time, second_cdc_operation_time, condition_str): config_id for config_id in config_ids}
        
        # 等待任务完成并获取结果
        for future in as_completed(future_to_success):
            config_id = future_to_success[future]
            try:
                result = future.result()
                logging.info(f'The task {config_id} result is {result}')
            except Exception as exc:
                logging.error(f'!!!!!!!!!! The task {config_id} generated an exception: {exc}')
                logging.error(traceback.format_exc())
                
                except_list.append(config_id)
    
    if len(except_list) >0:
        raise ValueError(f"!!!!!!!!!! Presence task failure: {except_list} ")

In [0]:
# downstream config
config_table_name = get_execute_param("config_table_name", None)
config_table_condition_str = get_execute_param("config_table_condition_str", None)

config_id_list = get_execute_param("config_id_list", None)

max_task_num = int(get_execute_param("max_task_num", 1))

task_inner_parallelism = int(get_execute_param("task_inner_parallelism", 1))

first_cdc_operation_time = get_execute_param("first_cdc_operation_time", None)
second_cdc_operation_time = get_execute_param("second_cdc_operation_time", None)

# 覆盖配置表中数据过滤条件
condition_str = get_execute_param("condition_str", None)


# log config
step_num = dbutils.widgets.get("step_num")
project = dbutils.widgets.get("project")
task_id = dbutils.widgets.get("task_id")

with StepLogger(step_name="downstream_run", step_num=step_num, project=project, task_id=task_id) as logger:
    downstream_job_run(task_id, config_table_name, config_table_condition_str, config_id_list, max_task_num, task_inner_parallelism, first_cdc_operation_time, second_cdc_operation_time, condition_str)